# 🧮 Odometer Pilot: LLM Counting Failure Study — Qwen2.5

Appendix replication of the Llama-3.2 odometer study on the Qwen2.5 family.

| Model | Change MODEL_NAME to | n_layers |
|-------|---------------------|----------|
| Qwen2.5-1.5B-Instruct | `Qwen/Qwen2.5-1.5B-Instruct` | 28 |
| Qwen2.5-3B-Instruct | `Qwen/Qwen2.5-3B-Instruct` | 36 |
| Qwen2.5-7B-Instruct | `Qwen/Qwen2.5-7B-Instruct` | 28 |

**Only cell 1 (config) needs to change between model runs.**
All other cells are model-agnostic within the Qwen2.5 family.

## 0 · Install dependencies

In [1]:
%pip install transformers torch accelerate nbformat matplotlib scikit-learn --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.filterwarnings("ignore", message="Both `max_new_tokens`")

In [3]:
from huggingface_hub import login
login()

## 1 · Config

**THIS IS THE ONLY CELL TO CHANGE BETWEEN MODEL RUNS.**

```
Qwen2.5-1.5B-Instruct  ->  MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
Qwen2.5-3B-Instruct    ->  MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
Qwen2.5-7B-Instruct    ->  MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
```

CRITICAL_LAYERS is set as a placeholder — update it after running the logit lens
(cell 14) to the 5 layers surrounding the lock-in point for this model.

In [4]:
import re
import json
import random
from collections import Counter
from typing import Optional

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# ── CHANGE THIS PER RUN ───────────────────────────────────────────────────
# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
# MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Set after running logit lens (cell 14) — update to layers around lock-in point
# Placeholder values shown; replace with actual lock-in layer ± 2
# Example: if logit lens shows lock-in at L22, set [20, 21, 22, 23, 24]

CRITICAL_LAYERS = None  # set after logit lens
# CRITICAL_LAYERS = [20, 21, 22, 23, 24, 25, 26]

# ── Fixed config — do not change ─────────────────────────────────────────
N_RUNS         = 10
SEEDS          = list(range(N_RUNS))
TEMPERATURE    = 0.0
MAX_NEW_TOKENS = 16
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

# Short model tag for output filenames
MODEL_TAG = MODEL_NAME.split("/")[-1].lower().replace("-", "_")

print(f"Model    : {MODEL_NAME}")
print(f"Tag      : {MODEL_TAG}")
print(f"Device   : {DEVICE}")
print(f"N_RUNS   : {N_RUNS}, TEMPERATURE: {TEMPERATURE}")

Model    : Qwen/Qwen2.5-7B-Instruct
Tag      : qwen2.5_7b_instruct
Device   : cuda
N_RUNS   : 10, TEMPERATURE: 0.0


## 2 · Define prompts

In [5]:
PROMPTS = {
    "phase1_baseline": {
        "text": (
            'Count the number of times "apple" appears in this list: '
            "apple apple apple apple apple apple apple apple apple apple. "
            "Respond only with the integer, nothing else."
        ),
        "expected": 10,
        "description": "10 identical tokens (apple x10)",
    },
    "phase2_anomaly": {
        "text": (
            'Count the number of times "apple" appears in this list: '
            "apple apple apple apple banana apple apple apple apple apple. "
            "Respond only with the integer, nothing else."
        ),
        "expected": 9,
        "description": "9 apple + 1 banana at position 5",
    },
    "phase3_control": {
        "text": (
            "Count the number of words in this list: "
            "dog cat car red blue green house tree book pen. "
            "Respond only with the integer, nothing else."
        ),
        "expected": 10,
        "description": "10 unique tokens (distinct words)",
    },
}

# Fixed prompts — comma-separated, no 'comma-separated' in instruction wording
PROMPTS_FIXED = {
    "phase1_baseline": {
        "text": (
            'Count the number of times "apple" appears in this list: '
            "apple, apple, apple, apple, apple, apple, apple, apple, apple, apple. "
            "Respond only with the integer, nothing else."
        ),
        "expected": 10,
        "description": "10 identical tokens, comma-separated",
    },
    "phase2_anomaly": {
        "text": (
            'Count the number of times "apple" appears in this list: '
            "apple, apple, apple, apple, banana, apple, apple, apple, apple, apple. "
            "Respond only with the integer, nothing else."
        ),
        "expected": 9,
        "description": "9 apple + 1 banana at position 5, comma-separated",
    },
    "phase3_control": {
        "text": (
            "Count the number of words in this list: "
            "dog, cat, car, red, blue, green, house, tree, book, pen. "
            "Respond only with the integer, nothing else."
        ),
        "expected": 10,
        "description": "10 unique tokens, comma-separated",
    },
}

for k, v in PROMPTS.items():
    print(f"[{k}]\n  Expected : {v['expected']}\n  Prompt   : {v['text'][:80]}...\n")

[phase1_baseline]
  Expected : 10
  Prompt   : Count the number of times "apple" appears in this list: apple apple apple apple ...

[phase2_anomaly]
  Expected : 9
  Prompt   : Count the number of times "apple" appears in this list: apple apple apple apple ...

[phase3_control]
  Expected : 10
  Prompt   : Count the number of words in this list: dog cat car red blue green house tree bo...



## 3 · Load model & tokenizer

In [6]:
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
)

n_layers = model.config.num_hidden_layers
print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")
print(f"Layers: {n_layers}")
print(f"Hidden dim: {model.config.hidden_size}")

Loading Qwen/Qwen2.5-7B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded. Parameters: 7.62B
Layers: 28
Hidden dim: 3584


## 4 · Helpers

In [7]:
def extract_count(raw_output: str) -> Optional[int]:
    match = re.search(r'\b(\d+)\b', raw_output.strip())
    return int(match.group(1)) if match else None


def get_top_digit(logits_1d):
    """Find digit token (1-20) with highest logit. Handles single-token digits only."""
    candidates = {}
    for n in range(1, 21):
        ids = tokenizer.encode(str(n), add_special_tokens=False)
        if len(ids) == 1:
            candidates[str(n)] = logits_1d[ids[0]].item()
    return max(candidates, key=candidates.get) if candidates else "?"


def make_inputs(prompt_text):
    """Apply chat template and return input dict on model device."""
    messages = [{"role": "user", "content": prompt_text}]
    return tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)


def remove_all_hooks(m):
    for module in m.modules():
        module._forward_hooks.clear()
        module._forward_pre_hooks.clear()
        module._backward_hooks.clear()


# Sanity checks
assert extract_count("10") == 10
assert extract_count("The answer is 9.") == 9
assert extract_count("no number") is None

# Verify digit tokenization for this model
print("Digit token check (single-token digits only):")
for n in range(1, 16):
    ids = tokenizer.encode(str(n), add_special_tokens=False)
    tag = "OK" if len(ids) == 1 else f"MULTI-TOKEN ({ids})"
    print(f"  {n:>3} -> {ids} -> {tag}")

print("\nHelpers OK.")

Digit token check (single-token digits only):
    1 -> [16] -> OK
    2 -> [17] -> OK
    3 -> [18] -> OK
    4 -> [19] -> OK
    5 -> [20] -> OK
    6 -> [21] -> OK
    7 -> [22] -> OK
    8 -> [23] -> OK
    9 -> [24] -> OK
   10 -> [16, 15] -> MULTI-TOKEN ([16, 15])
   11 -> [16, 16] -> MULTI-TOKEN ([16, 16])
   12 -> [16, 17] -> MULTI-TOKEN ([16, 17])
   13 -> [16, 18] -> MULTI-TOKEN ([16, 18])
   14 -> [16, 19] -> MULTI-TOKEN ([16, 19])
   15 -> [16, 20] -> MULTI-TOKEN ([16, 20])

Helpers OK.


## 5 · Run experiment — original prompts

In [8]:
def run_single(phase_key, seed):
    entry  = PROMPTS[phase_key]
    prompt = entry["text"]

    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

    raw = pipe(
        [{"role": "user", "content": prompt}],
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        do_sample=False,
        pad_token_id=pipe.tokenizer.eos_token_id,
        return_full_text=False,
    )[0]["generated_text"].strip()

    predicted = extract_count(raw)
    correct   = (predicted == entry["expected"]) if predicted is not None else False
    return {"seed": seed, "raw": raw, "predicted": predicted,
            "expected": entry["expected"], "correct": correct}


all_results = {}

for phase_key, entry in PROMPTS.items():
    print(f"\n{chr(9472)*55}")
    print(f"  {phase_key.upper()}  |  {entry['description']}")
    print(f"{chr(9472)*55}")
    phase_results = []
    for seed in SEEDS:
        r      = run_single(phase_key, seed)
        status = "✓" if r["correct"] else "✗"
        print(f"  seed={seed:02d}  predicted={str(r['predicted']):>4}  "
              f"expected={r['expected']}  {status}  raw={repr(r['raw'][:40])}")
        phase_results.append(r)
    all_results[phase_key] = phase_results

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



───────────────────────────────────────────────────────
  PHASE1_BASELINE  |  10 identical tokens (apple x10)
───────────────────────────────────────────────────────


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  seed=00  predicted=  10  expected=10  ✓  raw='10'
  seed=01  predicted=  10  expected=10  ✓  raw='10'
  seed=02  predicted=  10  expected=10  ✓  raw='10'


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  seed=00  predicted=  10  expected=10  ✓  raw='10'
  seed=01  predicted=  10  expected=10  ✓  raw='10'
  seed=02  predicted=  10  expected=10  ✓  raw='10'


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  seed=03  predicted=  10  expected=10  ✓  raw='10'
  seed=04  predicted=  10  expected=10  ✓  raw='10'
  seed=05  predicted=  10  expected=10  ✓  raw='10'


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  seed=06  predicted=  10  expected=10  ✓  raw='10'
  seed=07  predicted=  10  expected=10  ✓  raw='10'
  seed=08  predicted=  10  expected=10  ✓  raw='10'
  seed=09  predicted=  10  expected=10  ✓  raw='10'


## 6 · Summary statistics

In [9]:
print(f"{'Phase':<25} {'Accuracy':>10}  Distribution of predictions")
print(chr(9472) * 65)
for phase_key, results in all_results.items():
    acc   = sum(r["correct"] for r in results) / N_RUNS
    dist  = Counter(str(r["predicted"]) for r in results)
    print(f"{phase_key:<25} {acc:>9.0%}  {dict(dist)}")

# Verify raw outputs
print("\nRaw generation check (first 3 seeds per phase):")
for phase_key, results in all_results.items():
    print(f"\n[{phase_key}]")
    for r in results[:3]:
        print(f"  seed={r['seed']}  raw={repr(r['raw'])}  predicted={r['predicted']}")

Phase                       Accuracy  Distribution of predictions
─────────────────────────────────────────────────────────────────
phase1_baseline                100%  {'10': 10}
phase2_anomaly                   0%  {'8': 10}
phase3_control                 100%  {'10': 10}

Raw generation check (first 3 seeds per phase):

[phase1_baseline]
  seed=0  raw='10'  predicted=10
  seed=1  raw='10'  predicted=10
  seed=2  raw='10'  predicted=10

[phase2_anomaly]
  seed=0  raw='8'  predicted=8
  seed=1  raw='8'  predicted=8
  seed=2  raw='8'  predicted=8

[phase3_control]
  seed=0  raw='10'  predicted=10
  seed=1  raw='10'  predicted=10
  seed=2  raw='10'  predicted=10


## 7 · Save original results

In [10]:
save_path = f"odometer_results_{MODEL_TAG}.json"
with open(save_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"Saved to {save_path}")

Saved to odometer_results_qwen2.5_7b_instruct.json


## 8 · Tokenization diagnostic

In [11]:
print("Tokenization diagnostic")
print("=" * 70)

for phase_key, entry in PROMPTS.items():
    payload = entry["text"].split(": ")[1].split(". Respond")[0]
    toks    = tokenizer.encode(payload, add_special_tokens=False)
    decoded = [tokenizer.decode([t]) for t in toks]
    match   = "✓ token count == word count" if len(toks) == len(payload.split()) \
              else f"✗ MISMATCH: {len(toks)} tokens vs {len(payload.split())} words"
    print(f"\n[{phase_key}]")
    print(f"  Payload      : {payload}")
    print(f"  Word count   : {len(payload.split())}")
    print(f"  Token count  : {len(toks)}")
    print(f"  Tokens       : {decoded}")
    print(f"  {match}")

Tokenization diagnostic

[phase1_baseline]
  Payload      : apple apple apple apple apple apple apple apple apple apple
  Word count   : 10
  Token count  : 10
  Tokens       : ['apple', ' apple', ' apple', ' apple', ' apple', ' apple', ' apple', ' apple', ' apple', ' apple']
  ✓ token count == word count

[phase2_anomaly]
  Payload      : apple apple apple apple banana apple apple apple apple apple
  Word count   : 10
  Token count  : 10
  Tokens       : ['apple', ' apple', ' apple', ' apple', ' banana', ' apple', ' apple', ' apple', ' apple', ' apple']
  ✓ token count == word count

[phase3_control]
  Payload      : dog cat car red blue green house tree book pen
  Word count   : 10
  Token count  : 10
  Tokens       : ['dog', ' cat', ' car', ' red', ' blue', ' green', ' house', ' tree', ' book', ' pen']
  ✓ token count == word count


## 9 · Fixed prompts — spot check then run

In [12]:
# Spot check first — if any phase fails here, adjust PROMPTS_FIXED wording
# before running the full fixed-prompt experiment below.
# Common issue: 'comma-separated' in P3 instruction causes overcounting in some models.
# Fix: remove that phrase from the P3 text (already done in PROMPTS_FIXED above).

print("Spot check fixed prompts:")
for phase_key, entry in PROMPTS_FIXED.items():
    raw = pipe(
        [{"role": "user", "content": entry["text"]}],
        max_new_tokens=8, temperature=0.0, do_sample=False,
        pad_token_id=tokenizer.eos_token_id, return_full_text=False,
    )[0]["generated_text"].strip()
    predicted = extract_count(raw)
    correct   = "✓" if predicted == entry["expected"] else "✗"
    print(f"  {phase_key:<25}  expected={entry['expected']}  got={predicted}  {correct}")

print("\nIf any phase shows ✗ above, adjust PROMPTS_FIXED wording before continuing.")

Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Spot check fixed prompts:
  phase1_baseline            expected=10  got=10  ✓
  phase2_anomaly             expected=9  got=5  ✗
  phase3_control             expected=10  got=10  ✓

If any phase shows ✗ above, adjust PROMPTS_FIXED wording before continuing.


In [13]:
# Check original prompts on Qwen2.5-7B before deciding what to fix
print("Original space-separated results:")
for phase_key, entry in PROMPTS.items():
    raw = pipe(
        [{"role": "user", "content": entry["text"]}],
        max_new_tokens=8, temperature=0.0, do_sample=False,
        pad_token_id=tokenizer.eos_token_id, return_full_text=False,
    )[0]["generated_text"].strip()
    predicted = extract_count(raw)
    correct   = "✓" if predicted == entry["expected"] else "✗"
    print(f"  {phase_key:<25}  expected={entry['expected']}  got={predicted}  {correct}")

Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original space-separated results:
  phase1_baseline            expected=10  got=10  ✓
  phase2_anomaly             expected=9  got=8  ✗
  phase3_control             expected=10  got=10  ✓


In [14]:
# Full behavioral for Qwen 7B — original prompts
all_results = {}

for phase_key, entry in PROMPTS.items():
    print(f"\n{chr(9472)*55}")
    print(f"  {phase_key.upper()}")
    print(f"{chr(9472)*55}")
    phase_results = []
    for seed in SEEDS:
        torch.manual_seed(seed)
        random.seed(seed)
        np.random.seed(seed)
        raw = pipe(
            [{"role": "user", "content": entry["text"]}],
            max_new_tokens=MAX_NEW_TOKENS, temperature=0.0, do_sample=False,
            pad_token_id=tokenizer.eos_token_id, return_full_text=False,
        )[0]["generated_text"].strip()
        predicted = extract_count(raw)
        correct   = (predicted == entry["expected"]) if predicted is not None else False
        status    = "✓" if correct else "✗"
        print(f"  seed={seed:02d}  predicted={str(predicted):>4}  "
              f"expected={entry['expected']}  {status}")
        phase_results.append({"seed": seed, "raw": raw, "predicted": predicted,
                               "expected": entry["expected"], "correct": correct})
    all_results[phase_key] = phase_results

print(f"\n{'Phase':<25} {'Accuracy':>10}  Distribution")
print(chr(9472) * 55)
for phase_key, results in all_results.items():
    acc  = sum(r["correct"] for r in results) / N_RUNS
    dist = Counter(str(r["predicted"]) for r in results)
    print(f"{phase_key:<25} {acc:>9.0%}  {dict(dist)}")

Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



───────────────────────────────────────────────────────
  PHASE1_BASELINE
───────────────────────────────────────────────────────
  seed=00  predicted=  10  expected=10  ✓


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  seed=01  predicted=  10  expected=10  ✓
  seed=02  predicted=  10  expected=10  ✓
  seed=03  predicted=  10  expected=10  ✓


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  seed=01  predicted=  10  expected=10  ✓
  seed=02  predicted=  10  expected=10  ✓
  seed=03  predicted=  10  expected=10  ✓


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  seed=04  predicted=  10  expected=10  ✓
  seed=05  predicted=  10  expected=10  ✓
  seed=06  predicted=  10  expected=10  ✓


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  seed=07  predicted=  10  expected=10  ✓
  seed=08  predicted=  10  expected=10  ✓
  seed=09  predicted=  10  expected=10  ✓

Phase                       Accuracy  Distribution
───────────────────────────────────────────────────────
phase1_baseline                100%  {'10': 10}
phase2_anomaly                   0%  {'8': 10}
phase3_control                 100%  {'10': 10}


In [15]:
# Run full fixed-prompt experiment (10 seeds)
all_results_fixed = {}

for phase_key, entry in PROMPTS_FIXED.items():
    print(f"\n{chr(9472)*55}")
    print(f"  {phase_key.upper()} [FIXED]  |  {entry['description']}")
    print(f"{chr(9472)*55}")
    phase_results = []
    for seed in SEEDS:
        torch.manual_seed(seed)
        random.seed(seed)
        np.random.seed(seed)
        raw = pipe(
            [{"role": "user", "content": entry["text"]}],
            max_new_tokens=MAX_NEW_TOKENS, temperature=0.0, do_sample=False,
            pad_token_id=tokenizer.eos_token_id, return_full_text=False,
        )[0]["generated_text"].strip()
        predicted = extract_count(raw)
        correct   = (predicted == entry["expected"]) if predicted is not None else False
        status    = "✓" if correct else "✗"
        print(f"  seed={seed:02d}  predicted={str(predicted):>4}  "
              f"expected={entry['expected']}  {status}")
        phase_results.append({"seed": seed, "raw": raw, "predicted": predicted,
                               "expected": entry["expected"], "correct": correct})
    all_results_fixed[phase_key] = phase_results

# Accuracy comparison
print("ACCURACY COMPARISON: ORIGINAL vs FIXED PROMPTS")
print("=" * 70)
print(f"  {'Phase':<25} {'Original':>10} {'Fixed':>10}  {'Delta':>8}")
print("  " + "-" * 58)
for phase_key in PROMPTS:
    orig  = sum(r["correct"] for r in all_results[phase_key]) / N_RUNS
    fixed = sum(r["correct"] for r in all_results_fixed[phase_key]) / N_RUNS
    print(f"  {phase_key:<25} {orig:>10.0%} {fixed:>10.0%}  {fixed-orig:>+8.0%}")

Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



───────────────────────────────────────────────────────
  PHASE1_BASELINE [FIXED]  |  10 identical tokens, comma-separated
───────────────────────────────────────────────────────
  seed=00  predicted=  10  expected=10  ✓


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  seed=01  predicted=  10  expected=10  ✓
  seed=02  predicted=  10  expected=10  ✓
  seed=03  predicted=  10  expected=10  ✓


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  seed=01  predicted=  10  expected=10  ✓
  seed=02  predicted=  10  expected=10  ✓
  seed=03  predicted=  10  expected=10  ✓


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  seed=04  predicted=  10  expected=10  ✓
  seed=05  predicted=  10  expected=10  ✓
  seed=06  predicted=  10  expected=10  ✓


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  seed=07  predicted=  10  expected=10  ✓
  seed=08  predicted=  10  expected=10  ✓
  seed=09  predicted=  10  expected=10  ✓
ACCURACY COMPARISON: ORIGINAL vs FIXED PROMPTS
  Phase                       Original      Fixed     Delta
  ----------------------------------------------------------
  phase1_baseline                 100%       100%       +0%
  phase2_anomaly                    0%         0%       +0%
  phase3_control                  100%       100%       +0%


## Anomaly sweep

In [16]:
# Anomaly sweep for Qwen 7B — does behavior match 3B?
print("=" * 65)
print("ANOMALY DETECTION SWEEP — Qwen2.5-7B-Instruct")
print("=" * 65)

print("\n[Test 1] Vary banana position")
print(f"  {'Position':>10}  {'Expected':>9}  {'Output':>8}  {'Detected?':>10}")
print("  " + "-" * 44)
for pos in range(10):
    words  = ["apple"] * 10
    words[pos] = "banana"
    prompt = (
        'Count the number of times "apple" appears in this list: '
        + " ".join(words)
        + ". Respond only with the integer, nothing else."
    )
    raw       = pipe([{"role": "user", "content": prompt}],
                     max_new_tokens=8, temperature=0.0, do_sample=False,
                     pad_token_id=tokenizer.eos_token_id,
                     return_full_text=False)[0]["generated_text"].strip()
    predicted = extract_count(raw)
    detected  = "YES ✓" if predicted == 9 else "NO ✗"
    print(f"  {pos:>10}  {9:>9}  {str(predicted):>8}  {detected:>10}")

print("\n[Test 2] Vary number of bananas")
print(f"  {'N bananas':>10}  {'Expected':>9}  {'Output':>8}  {'Detected?':>10}")
print("  " + "-" * 44)
for n_bananas in range(1, 6):
    words    = ["banana"] * n_bananas + ["apple"] * (10 - n_bananas)
    expected = 10 - n_bananas
    prompt   = (
        'Count the number of times "apple" appears in this list: '
        + " ".join(words)
        + ". Respond only with the integer, nothing else."
    )
    raw       = pipe([{"role": "user", "content": prompt}],
                     max_new_tokens=8, temperature=0.0, do_sample=False,
                     pad_token_id=tokenizer.eos_token_id,
                     return_full_text=False)[0]["generated_text"].strip()
    predicted = extract_count(raw)
    detected  = "YES ✓" if predicted == expected else "NO ✗"
    print(f"  {n_bananas:>10}  {expected:>9}  {str(predicted):>8}  {detected:>10}")

print("\n[Test 3] Edge cases")
edge_cases = [
    ("all bananas",   " ".join(["banana"] * 10), 0),
    ("all apples",    " ".join(["apple"]  * 10), 10),
    ("1 apple only",  " ".join(["banana"] * 9 + ["apple"]), 1),
]
for label, word_str, expected in edge_cases:
    prompt    = (
        'Count the number of times "apple" appears in this list: '
        + word_str
        + ". Respond only with the integer, nothing else."
    )
    raw       = pipe([{"role": "user", "content": prompt}],
                     max_new_tokens=8, temperature=0.0, do_sample=False,
                     pad_token_id=tokenizer.eos_token_id,
                     return_full_text=False)[0]["generated_text"].strip()
    predicted = extract_count(raw)
    detected  = "YES ✓" if predicted == expected else "NO ✗"
    print(f"  {label:<20}  expected={expected}  got={predicted}  {detected}")

Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANOMALY DETECTION SWEEP — Qwen2.5-7B-Instruct

[Test 1] Vary banana position
    Position   Expected    Output   Detected?
  --------------------------------------------
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
           0          9        10        NO ✗


Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


           1          9        10        NO ✗
           2          9        10        NO ✗
           3          9        10        NO ✗


Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

           8          9         8        NO ✗
           9          9         9       YES ✓

[Test 2] Vary number of bananas
   N bananas   Expected    Output   Detected?
  --------------------------------------------
           1          9        10        NO ✗


Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


           2          8         8       YES ✓
           3          7         7       YES ✓
           4          6         6       YES ✓
           5          5         5       YES ✓

[Test 3] Edge cases


Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  all bananas           expected=0  got=0  YES ✓
  all apples            expected=10  got=10  YES ✓
  1 apple only          expected=1  got=1  YES ✓


## 10 · Behavioral n-sweep

In [17]:
unique_vocab = ["dog", "cat", "car", "red", "blue", "green",
                "house", "tree", "book", "pen", "fish", "cup",
                "hat", "sun", "moon", "sky", "fire", "rain", "snow", "wind"]

NS = [5, 6, 7, 8, 9, 10, 11, 12, 15, 20]

print(f"BEHAVIORAL N-SWEEP — {MODEL_NAME}")
print("=" * 65)
print(f"  {'n':>4}  {'P1 output':>10}  {'P1 correct':>11}  "
      f"{'P3 output':>10}  {'P3 correct':>11}")
print("  " + "-" * 55)

sweep_behavioral = {}

for n in NS:
    prompt_p1 = (
        f'Count the number of times "apple" appears in this list: '
        + " ".join(["apple"] * n)
        + ". Respond only with the integer, nothing else."
    )
    prompt_p3 = (
        "Count the number of words in this list: "
        + " ".join(unique_vocab[:n])
        + ". Respond only with the integer, nothing else."
    )
    raw_p1 = pipe([{"role": "user", "content": prompt_p1}],
                  max_new_tokens=8, temperature=0.0, do_sample=False,
                  pad_token_id=tokenizer.eos_token_id,
                  return_full_text=False)[0]["generated_text"].strip()
    raw_p3 = pipe([{"role": "user", "content": prompt_p3}],
                  max_new_tokens=8, temperature=0.0, do_sample=False,
                  pad_token_id=tokenizer.eos_token_id,
                  return_full_text=False)[0]["generated_text"].strip()

    pred_p1 = extract_count(raw_p1)
    pred_p3 = extract_count(raw_p3)
    corr_p1 = "✓" if pred_p1 == n else "✗"
    corr_p3 = "✓" if pred_p3 == n else "✗"
    print(f"  {n:>4}  {str(pred_p1):>10}  {corr_p1:>11}  "
          f"{str(pred_p3):>10}  {corr_p3:>11}")
    sweep_behavioral[n] = {
        "n": n,
        "p1_output": pred_p1, "p1_correct": pred_p1 == n,
        "p3_output": pred_p3, "p3_correct": pred_p3 == n,
    }

print("\nP1 attractor pattern:")
for n, r in sweep_behavioral.items():
    marker = "= n (correct)" if r["p1_correct"] else f"≠ n (wrong, got {r['p1_output']})"
    print(f"  n={n:>2}  output={r['p1_output']}  {marker}")

with open(f"behavioral_n_sweep_{MODEL_TAG}.json", "w") as f:
    json.dump({str(k): v for k, v in sweep_behavioral.items()}, f, indent=2)
print(f"\nSaved: behavioral_n_sweep_{MODEL_TAG}.json")

Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BEHAVIORAL N-SWEEP — Qwen/Qwen2.5-7B-Instruct
     n   P1 output   P1 correct   P3 output   P3 correct
  -------------------------------------------------------
     5           5            ✓           5            ✓


Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_ne

     6           6            ✓           6            ✓
     7           7            ✓           7            ✓


Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

    10          10            ✓          10            ✓
    11          11            ✓          12            ✗


Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    12          12            ✓          12            ✓
    15          16            ✗          12            ✗


Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    20          20            ✓          16            ✗

P1 attractor pattern:
  n= 5  output=5  = n (correct)
  n= 6  output=6  = n (correct)
  n= 7  output=7  = n (correct)
  n= 8  output=8  = n (correct)
  n= 9  output=9  = n (correct)
  n=10  output=10  = n (correct)
  n=11  output=11  = n (correct)
  n=12  output=12  = n (correct)
  n=15  output=16  ≠ n (wrong, got 16)
  n=20  output=20  = n (correct)

Saved: behavioral_n_sweep_qwen2.5_7b_instruct.json


## 11 · Prompt paraphrase robustness

In [18]:
PARAPHRASES = {
    "original": (
        'Count the number of times "apple" appears in this list: '
        "{list}. Respond only with the integer, nothing else."
    ),
    "how_many": (
        'How many times does the word "apple" appear in the following list: '
        "{list}? Answer with a single integer, nothing else."
    ),
    "list_first": (
        "List: {list}\n"
        'How many times does "apple" appear? Single integer only.'
    ),
    "tally": (
        'Tally the occurrences of "apple" in this sequence: '
        "{list}. Output only the count as an integer."
    ),
    "simple": (
        'Count "apple" in: {list}. '
        "Reply with just the number."
    ),
}

APPLE_LIST_10         = " ".join(["apple"] * 10)
APPLE_LIST_10_ANOMALY = "apple apple apple apple banana apple apple apple apple apple"
UNIQUE_LIST_10        = "dog cat car red blue green house tree book pen"

test_cases = {
    "P1_repeated": (APPLE_LIST_10,         10),
    "P2_anomaly" : (APPLE_LIST_10_ANOMALY,  9),
    "P3_unique"  : (UNIQUE_LIST_10,         10),
}

print(f"PROMPT PARAPHRASE ROBUSTNESS — {MODEL_NAME}")
print("=" * 75)
print(f"  {'Paraphrase':<15}", end="")
for case in test_cases:
    print(f"  {case:<16}", end="")
print()
print("  " + "-" * 68)

paraphrase_results = {}

for pname, template in PARAPHRASES.items():
    print(f"  {pname:<15}", end="")
    paraphrase_results[pname] = {}
    for case_name, (word_list, expected) in test_cases.items():
        if case_name == "P3_unique":
            prompt = ("Count the number of words in this list: "
                      f"{word_list}. Respond only with the integer, nothing else.")
        else:
            prompt = template.format(list=word_list)
        raw = pipe([{"role": "user", "content": prompt}],
                   max_new_tokens=8, temperature=0.0, do_sample=False,
                   pad_token_id=tokenizer.eos_token_id,
                   return_full_text=False)[0]["generated_text"].strip()
        predicted = extract_count(raw)
        correct   = "✓" if predicted == expected else "✗"
        print(f"  {str(predicted)+'('+correct+')' :<16}", end="")
        paraphrase_results[pname][case_name] = {
            "predicted": predicted, "expected": expected,
            "correct": predicted == expected,
        }
    print()

print("\nP1 attractor stability:")
p1_preds = [paraphrase_results[p]["P1_repeated"]["predicted"] for p in PARAPHRASES]
modal    = Counter(p1_preds).most_common(1)[0][0]
for pname in PARAPHRASES:
    pred = paraphrase_results[pname]["P1_repeated"]["predicted"]
    tag  = "STABLE" if pred == modal else f"SHIFTED from {modal}"
    print(f"  {pname:<15}  predicted={pred}  {tag}")

with open(f"paraphrase_robustness_{MODEL_TAG}.json", "w") as f:
    json.dump(paraphrase_results, f, indent=2)
print(f"\nSaved: paraphrase_robustness_{MODEL_TAG}.json")

Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT PARAPHRASE ROBUSTNESS — Qwen/Qwen2.5-7B-Instruct
  Paraphrase       P1_repeated       P2_anomaly        P3_unique       
  --------------------------------------------------------------------
  original         10(✓)             8(✗)            

Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  10(✓)           
  how_many         10(✓)             8(✗)            

Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  10(✓)           
  list_first       10(✓)             9(✓)            

Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  10(✓)           
  tally            10(✓)             5(✗)            

Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  10(✓)           
  simple           10(✓)             8(✗)              10(✓)           

P1 attractor stability:
  original         predicted=10  STABLE
  how_many         predicted=10  STABLE
  list_first       predicted=10  STABLE
  tally            predicted=10  STABLE
  simple           predicted=10  STABLE

Saved: paraphrase_robustness_qwen2.5_7b_instruct.json


## 12 · Load model_eager (for mechanistic cells)

Keep `model` (sdpa) alive for inference — it's faster.
`model_eager` is only needed for attention extraction and logit lens.

In [19]:
model_eager = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
model_eager.eval()
print(f"Eager model loaded.")
print(f"Layers     : {model_eager.config.num_hidden_layers}")
print(f"Attn impl  : {model_eager.config._attn_implementation}")

# Update make_inputs to use model_eager device for mechanistic cells
def make_inputs_eager(prompt_text):
    messages = [{"role": "user", "content": prompt_text}]
    return tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model_eager.device)

# Inspect layer submodule names — needed to confirm hook targets
# Qwen2.5 should match Llama (self_attn, mlp, input_layernorm, post_attention_layernorm)
# but verify here in case of version differences
print("\nLayer submodule names:")
for name, _ in model_eager.model.layers[0].named_children():
    print(f"  {name}")
print("\nExpected: self_attn, mlp, input_layernorm, post_attention_layernorm")
print("If different, update hook targets in cells 15 and 16.")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Eager model loaded.
Layers     : 28
Attn impl  : eager

Layer submodule names:
  self_attn
  mlp
  input_layernorm
  post_attention_layernorm

Expected: self_attn, mlp, input_layernorm, post_attention_layernorm
If different, update hook targets in cells 15 and 16.


## 13 · Attention analysis

In [20]:
def get_attentions(prompt_text):
    inputs = make_inputs_eager(prompt_text)
    with torch.no_grad():
        out = model_eager(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            output_attentions=True,
        )
    attentions = torch.stack(out.attentions).squeeze(1)  # (n_layers, n_heads, seq, seq)
    return attentions, inputs["input_ids"][0]


def find_word_positions(tokens, phase_key):
    content_words = {
        "phase1_baseline": {"apple"},
        "phase2_anomaly" : {"apple", "banana"},
        "phase3_control" : {"dog", "cat", "car", "red", "blue",
                            "green", "house", "tree", "book", "pen"},
    }
    valid = content_words[phase_key]
    colon_idx = max(i for i, t in enumerate(tokens) if t.strip() == ":")
    positions = []
    for i in range(colon_idx + 1, len(tokens)):
        if tokens[i].strip() in valid:
            positions.append(i)
        elif "." in tokens[i] and positions:
            break
    return positions


print("=" * 70)
print("ATTENTION ANALYSIS — word-list tokens only")
print("=" * 70)

attn_summary = {}

for phase_key in ["phase1_baseline", "phase2_anomaly", "phase3_control"]:
    attentions, input_ids = get_attentions(PROMPTS[phase_key]["text"])
    tokens        = [tokenizer.decode([t]) for t in input_ids]
    word_positions = find_word_positions(tokens, phase_key)
    word_tokens    = [tokens[p] for p in word_positions]

    per_head  = attentions.cpu().float().numpy()            # (n_layers, n_heads, seq, seq)
    word_attn = per_head[:, :, -1, :][:, :, word_positions] # last token -> word positions
    word_attn_norm = word_attn / (word_attn.sum(axis=-1, keepdims=True) + 1e-9)
    mean_word_attn = word_attn_norm.mean(axis=1)             # (n_layers, n_words)

    entropies = []
    for l in range(mean_word_attn.shape[0]):
        a = mean_word_attn[l] / (mean_word_attn[l].sum() + 1e-9)
        entropies.append(round(float(-(a * np.log(a + 1e-9)).sum()), 4))

    uniformity = (mean_word_attn.min(axis=-1) /
                  (mean_word_attn.max(axis=-1) + 1e-9)).mean()

    print(f"\n[{phase_key}]")
    print(f"  Word positions : {word_positions}")
    print(f"  Word tokens    : {word_tokens}")
    print(f"  Mean entropy   : {np.mean(entropies):.4f}")
    print(f"  Uniformity     : {float(uniformity):.4f}")

    attn_summary[phase_key] = {
        "word_positions": word_positions, "word_tokens": word_tokens,
        "entropy_per_layer": entropies, "mean_uniformity": float(uniformity),
    }

with open(f"attention_analysis_{MODEL_TAG}.json", "w") as f:
    json.dump(attn_summary, f, indent=2)
print(f"\nSaved: attention_analysis_{MODEL_TAG}.json")

ATTENTION ANALYSIS — word-list tokens only

[phase1_baseline]
  Word positions : [37, 38, 39, 40, 41, 42, 43, 44, 45, 46]
  Word tokens    : [' apple', ' apple', ' apple', ' apple', ' apple', ' apple', ' apple', ' apple', ' apple', ' apple']
  Mean entropy   : 2.1271
  Uniformity     : 0.2591

[phase2_anomaly]
  Word positions : [37, 38, 39, 40, 41, 42, 43, 44, 45, 46]
  Word tokens    : [' apple', ' apple', ' apple', ' apple', ' banana', ' apple', ' apple', ' apple', ' apple', ' apple']
  Mean entropy   : 2.1234
  Uniformity     : 0.2154

[phase3_control]
  Word positions : [33, 34, 35, 36, 37, 38, 39, 40, 41, 42]
  Word tokens    : [' dog', ' cat', ' car', ' red', ' blue', ' green', ' house', ' tree', ' book', ' pen']
  Mean entropy   : 2.1395
  Uniformity     : 0.2150

Saved: attention_analysis_qwen2.5_7b_instruct.json


## 14 · Linear probes

In [21]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error, r2_score


def get_hidden_states(prompt_text):
    inputs = make_inputs_eager(prompt_text)
    with torch.no_grad():
        out = model_eager(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            output_hidden_states=True,
        )
    hidden = torch.stack(out.hidden_states)  # (n_layers+1, 1, seq_len, hidden_dim)
    return hidden[:, 0, -1, :].cpu().float().numpy()  # (n_layers+1, hidden_dim)


def run_loo_probe(hidden_array, labels):
    loo  = LeaveOneOut()
    maes, r2s = [], []
    for layer_idx in range(hidden_array.shape[1]):
        X      = StandardScaler().fit_transform(hidden_array[:, layer_idx, :])
        preds  = np.zeros(len(labels))
        for train_idx, test_idx in loo.split(X):
            clf = Ridge(alpha=1.0)
            clf.fit(X[train_idx], labels[train_idx])
            preds[test_idx] = clf.predict(X[test_idx])
        maes.append(mean_absolute_error(labels, preds))
        r2s.append(r2_score(labels, preds))
    return maes, r2s


probe_ns    = list(range(3, 14))
labels      = np.array(probe_ns, dtype=float)

repeated_prompts = [
    f'Count the number of times "apple" appears in this list: '
    + " ".join(["apple"] * n)
    + ". Respond only with the integer, nothing else."
    for n in probe_ns
]
unique_prompts = [
    "Count the number of words in this list: "
    + " ".join(unique_vocab[:n])
    + ". Respond only with the integer, nothing else."
    for n in probe_ns
]

print("Collecting activations...")
hidden_repeated = np.stack([get_hidden_states(p) for p in repeated_prompts])
hidden_unique   = np.stack([get_hidden_states(p) for p in unique_prompts])

print("Running probes...")
maes_rep,  r2s_rep  = run_loo_probe(hidden_repeated, labels)
maes_uniq, r2s_uniq = run_loo_probe(hidden_unique,   labels)

print(f"\n{'Layer':>6}  {'MAE(rep)':>10}  {'R2(rep)':>9}  "
      f"{'MAE(uniq)':>10}  {'R2(uniq)':>9}  {'ΔMAE':>8}")
print("-" * 62)
n_layers_plus1 = hidden_repeated.shape[1]
for i in range(n_layers_plus1):
    label = "embed" if i == 0 else f"L{i:02d}"
    delta = maes_rep[i] - maes_uniq[i]
    print(f"{label:>6}  {maes_rep[i]:>10.4f}  {r2s_rep[i]:>9.4f}  "
          f"{maes_uniq[i]:>10.4f}  {r2s_uniq[i]:>9.4f}  {delta:>+8.4f}")

probe_results = {
    "ns": probe_ns, "labels": probe_ns,
    "repeated": {"maes": maes_rep,  "r2s": r2s_rep},
    "unique"  : {"maes": maes_uniq, "r2s": r2s_uniq},
}
with open(f"probe_results_{MODEL_TAG}.json", "w") as f:
    json.dump(probe_results, f, indent=2)
print(f"\nSaved: probe_results_{MODEL_TAG}.json")

Running probes...

 Layer    MAE(rep)    R2(rep)   MAE(uniq)   R2(uniq)      ΔMAE
--------------------------------------------------------------
 embed      3.0000    -0.2100      3.0000    -0.2100   +0.0000
   L01      0.3151     0.9685      0.7140     0.9023   -0.3990
   L02      0.3942     0.9565      0.7619     0.8812   -0.3678
   L03      0.4126     0.9425      0.6971     0.8938   -0.2845
   L04      0.4124     0.9489      0.6351     0.9101   -0.2226
   L05      0.3923     0.9467      0.6596     0.8997   -0.2673
   L06      0.3404     0.9558      0.4960     0.9526   -0.1555
   L07      0.3555     0.9606      0.4468     0.9633   -0.0913
   L08      0.4041     0.9542      0.3912     0.9757   +0.0129
   L09      0.4279     0.9554      0.3645     0.9782   +0.0634
   L10      0.3888     0.9647      0.3048     0.9803   +0.0841
   L11      0.4232     0.9648      0.3413     0.9763   +0.0819
   L12      0.4305     0.9618      0.3173     0.9733   +0.1132
   L13      0.4392     0.9611      0

## 15 · Logit lens

**After running this cell:**
1. Identify the layer where the wrong answer locks in for P1
2. Go back to cell 1 and set `CRITICAL_LAYERS` to that layer ± 2
3. Then run cells 16 and 17

In [22]:
def logit_lens_full(prompt_text, top_k=5):
    inputs = make_inputs_eager(prompt_text)
    with torch.no_grad():
        out = model_eager(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            output_hidden_states=True,
        )
    unembed = model_eager.lm_head.weight
    norm    = model_eager.model.norm
    results = []
    for layer_idx, h in enumerate(out.hidden_states):
        last        = h[0, -1, :]
        last_normed = norm(last.unsqueeze(0).unsqueeze(0)).squeeze()
        logits      = unembed @ last_normed
        top_digit   = get_top_digit(logits)
        top5        = [tokenizer.decode([i]) for i in logits.topk(top_k).indices]
        results.append({
            "layer"     : "embed" if layer_idx == 0 else f"L{layer_idx:02d}",
            "top_digit" : top_digit,
            "top5"      : top5,
        })
    return results


print("=" * 70)
print(f"LOGIT LENS — {MODEL_NAME}")
print(f"Total layers: {model_eager.config.num_hidden_layers}")
print("=" * 70)

lens_results = {}

for phase_key in ["phase1_baseline", "phase3_control"]:
    correct = PROMPTS[phase_key]["expected"]
    print(f"\n[{phase_key}]  correct={correct}")
    print(f"  {'Layer':>6}  {'Top digit':>10}  Top-5 tokens")
    print("  " + "-" * 55)
    lens = logit_lens_full(PROMPTS[phase_key]["text"])
    lens_results[phase_key] = lens
    for r in lens:
        print(f"  {r['layer']:>6}  {r['top_digit']:>10}  {r['top5']}")

# Normalized depth summary
n_layers = model_eager.config.num_hidden_layers
print(f"\n{'='*50}")
print("LOCK-IN SUMMARY")
print(f"{'='*50}")
print(f"Total layers: {n_layers}")
print(f"Equivalent of 87.5% depth (Llama-1B lock-in): L{round(0.875 * n_layers)}")
print("\n>>> After reading P1 output above:")
print(">>> Go to cell 1, set CRITICAL_LAYERS = [lock_in_layer-2 .. lock_in_layer+2]")
print(">>> Then run cells 16 and 17")

with open(f"logit_lens_{MODEL_TAG}.json", "w") as f:
    json.dump(lens_results, f, indent=2)
print(f"\nSaved: logit_lens_{MODEL_TAG}.json")

LOGIT LENS — Qwen/Qwen2.5-7B-Instruct
Total layers: 28

[phase1_baseline]  correct=10
   Layer   Top digit  Top-5 tokens
  -------------------------------------------------------
   embed           9  ['rray', 'ystack', 'acements', 'emax', 'emade']
     L01           1  ['-strokes', ' strugg', '℠', ' WHETHER', ' algu']
     L02           1  ['-strokes', '换句话', '如果你想', ' algu', 'こともあります']
     L03           1  ['-strokes', '换句话', ' algu', ' bureaucr', ' strugg']
     L04           1  ['-strokes', '换句话', ' sophistic', ' 若要', ' bureaucr']
     L05           1  ['换句话', '-strokes', '从根本', '如果你想', ' 若要']
     L06           4  ['换句话', '-strokes', '/Peak', ' licensors', ' strugg']
     L07           4  ['换句话', '/Peak', ' bureaucr', '-strokes', ' strugg']
     L08           4  ['换句话', ' strugg', '-strokes', ' libertine', ' bureaucr']
     L09           1  ['换句话', '/Dk', ' libertine', '-strokes', ' strugg']
     L10           1  ['换句话', ' strugg', '-strokes', ' libertine', ' bureaucr']
     L11 

In [23]:
# Run full behavioral experiment to confirm
all_results = {}

for phase_key, entry in PROMPTS.items():
    print(f"\n{chr(9472)*55}")
    print(f"  {phase_key.upper()}")
    print(f"{chr(9472)*55}")
    phase_results = []
    for seed in SEEDS:
        torch.manual_seed(seed)
        random.seed(seed)
        np.random.seed(seed)
        raw = pipe(
            [{"role": "user", "content": entry["text"]}],
            max_new_tokens=MAX_NEW_TOKENS, temperature=0.0, do_sample=False,
            pad_token_id=tokenizer.eos_token_id, return_full_text=False,
        )[0]["generated_text"].strip()
        predicted = extract_count(raw)
        correct   = (predicted == entry["expected"]) if predicted is not None else False
        status    = "✓" if correct else "✗"
        print(f"  seed={seed:02d}  predicted={str(predicted):>4}  "
              f"expected={entry['expected']}  {status}")
        phase_results.append({"seed": seed, "raw": raw, "predicted": predicted,
                               "expected": entry["expected"], "correct": correct})
    all_results[phase_key] = phase_results

print(f"\n{'Phase':<25} {'Accuracy':>10}  Distribution")
print(chr(9472) * 55)
for phase_key, results in all_results.items():
    acc  = sum(r["correct"] for r in results) / N_RUNS
    dist = Counter(str(r["predicted"]) for r in results)
    print(f"{phase_key:<25} {acc:>9.0%}  {dict(dist)}")

Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



───────────────────────────────────────────────────────
  PHASE1_BASELINE
───────────────────────────────────────────────────────
  seed=00  predicted=  10  expected=10  ✓


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  seed=01  predicted=  10  expected=10  ✓
  seed=02  predicted=  10  expected=10  ✓
  seed=03  predicted=  10  expected=10  ✓


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  seed=01  predicted=  10  expected=10  ✓
  seed=02  predicted=  10  expected=10  ✓
  seed=03  predicted=  10  expected=10  ✓


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  seed=04  predicted=  10  expected=10  ✓
  seed=05  predicted=  10  expected=10  ✓
  seed=06  predicted=  10  expected=10  ✓


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  seed=07  predicted=  10  expected=10  ✓
  seed=08  predicted=  10  expected=10  ✓
  seed=09  predicted=  10  expected=10  ✓

Phase                       Accuracy  Distribution
───────────────────────────────────────────────────────
phase1_baseline                100%  {'10': 10}
phase2_anomaly                   0%  {'8': 10}
phase3_control                 100%  {'10': 10}


In [24]:
summary_qwen7b = {
    "model"          : MODEL_NAME,
    "n_layers"       : model.config.num_hidden_layers,
    "behavioral"     : {
        "P1": {"accuracy": "100%", "output": "10"},
        "P2": {"accuracy": "0%",   "output": "8"},
        "P3": {"accuracy": "100%", "output": "10"},
    },
    "n_sweep"        : "correct n=5-12, off-by-one at n=15, correct n=20",
    "anomaly_sweep"  : {
        "position_detection" : "detects banana at positions 6,7,9 — recency bias same as 3B",
        "quantity_threshold" : "detects anomaly when >=2 bananas — lower than 3B (>=5)",
        "single_banana_mid"  : "still fails for 1 banana at positions 0-5,8",
    },
    "scaling_note"   : (
        "Counting solved at 3B, near-perfect at 7B. "
        "Anomaly detection improves with scale (threshold 5->2 bananas) "
        "but single mid-sequence intruder detection not solved at 7B."
    ),
}
with open("summary_qwen2.5_7b.json", "w") as f:
    json.dump(summary_qwen7b, f, indent=2)
print("Saved: summary_qwen2.5_7b.json")
print("\nAll Qwen experiments complete.")
print("Qwen 1.5B: counting failure")
print("Qwen 3B:   counting solved, anomaly detection fails (threshold=5)")
print("Qwen 7B:   counting near-perfect, anomaly detection improves (threshold=2)")

Saved: summary_qwen2.5_7b.json

All Qwen experiments complete.
Qwen 1.5B: counting failure
Qwen 3B:   counting solved, anomaly detection fails (threshold=5)
Qwen 7B:   counting near-perfect, anomaly detection improves (threshold=2)


## Additional experiments

In [25]:
# Does the "8" attractor appear when prompt is in Chinese?
prompts_chinese = {
    "P1_chinese": '计算"苹果"在这个列表中出现的次数：苹果 苹果 苹果 苹果 苹果 苹果 苹果 苹果 苹果 苹果。只回答数字。',
    "P1_english": 'Count the number of times "apple" appears in this list: apple apple apple apple apple apple apple apple apple apple. Respond only with the integer, nothing else.',
}
for name, prompt in prompts_chinese.items():
    raw = pipe([{"role": "user", "content": prompt}],
               max_new_tokens=8, temperature=0.0, do_sample=False,
               pad_token_id=tokenizer.eos_token_id,
               return_full_text=False)[0]["generated_text"].strip()
    print(f"{name}: {repr(raw)}")

Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


P1_chinese: '10'
P1_english: '10'


In [26]:
# Does the model count repeated digits correctly?
for symbol in ["1", "0", "7", "the"]:
    word_list = " ".join([symbol] * 10)
    prompt = (
        f'Count the number of times "{symbol}" appears in this list: '
        f"{word_list}. Respond only with the integer, nothing else."
    )
    raw = pipe([{"role": "user", "content": prompt}],
               max_new_tokens=8, temperature=0.0, do_sample=False,
               pad_token_id=tokenizer.eos_token_id,
               return_full_text=False)[0]["generated_text"].strip()
    print(f"symbol={repr(symbol):<8}  output={repr(raw)}")

Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


symbol='1'       output='10'
symbol='0'       output='10'
symbol='7'       output='10'
symbol='the'     output='10'


In [29]:
cot_prompt = (
    'Count the number of times "apple" appears in this list: '
    "apple apple apple apple apple apple apple apple apple apple. "
    "Think step by step, then give the final count as an integer on the last line."
)
raw = pipe([{"role": "user", "content": cot_prompt}],
           max_new_tokens=300, temperature=0.0, do_sample=False,
           pad_token_id=tokenizer.eos_token_id,
           return_full_text=False)[0]["generated_text"].strip()
print(repr(raw))

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Let\'s count the number of times "apple" appears in the given list step by step:\n\n1. The first item is "apple".\n2. The second item is "apple".\n3. The third item is "apple".\n4. The fourth item is "apple".\n5. The fifth item is "apple".\n6. The sixth item is "apple".\n7. The seventh item is "apple".\n8. The eighth item is "apple".\n9. The ninth item is "apple".\n10. The tenth item is "apple".\n\nAfter going through each item, we can see that "apple" appears 10 times.\n\nFinal count: 10'


In [28]:
# Compute steering vector: mean activation at L14 for correct prompts - incorrect prompts
# Correct: comma-separated (model gets 10)
# Incorrect: space-separated (model gets 8)
# Then add scaled steering vector during P1 forward pass and check if output changes

correct_prompt   = 'Count the number of times "apple" appears in this list: apple, apple, apple, apple, apple, apple, apple, apple, apple, apple. Respond only with the integer, nothing else.'
incorrect_prompt = 'Count the number of times "apple" appears in this list: apple apple apple apple apple apple apple apple apple apple. Respond only with the integer, nothing else.'

# Get L14 activations for both
cache_correct   = {}
cache_incorrect = {}

def make_cache_hook(cache):
    def hook(module, input, output):
        h = output[0] if isinstance(output, tuple) else output
        cache["h"] = h[0, -1, :].detach().clone()
    return hook

for prompt, cache in [(correct_prompt, cache_correct), 
                       (incorrect_prompt, cache_incorrect)]:
    remove_all_hooks(model_eager)
    h = model_eager.model.layers[13].register_forward_hook(make_cache_hook(cache))
    with torch.no_grad():
        model_eager(**make_inputs_eager(prompt))
    h.remove()

# Steering vector = correct - incorrect at L14
steering_vec = cache_correct["h"] - cache_incorrect["h"]
print(f"Steering vector norm: {steering_vec.norm().item():.4f}")

# Apply at different scales and check output
for alpha in [0.5, 1.0, 2.0, 5.0, 10.0]:
    def steer_hook(module, input, output):
        h = output[0] if isinstance(output, tuple) else output
        h = h.clone()
        h[0, -1, :] = h[0, -1, :] + alpha * steering_vec
        return (h,) + output[1:] if isinstance(output, tuple) else h

    remove_all_hooks(model_eager)
    handle = model_eager.model.layers[13].register_forward_hook(steer_hook)
    with torch.no_grad():
        out = model_eager(**make_inputs_eager(incorrect_prompt))
    handle.remove()

    td   = get_top_digit(out.logits[0, -1, :])
    top5 = [tokenizer.decode([i]) for i in out.logits[0, -1, :].topk(5).indices]
    print(f"  alpha={alpha:>5.1f}  top digit={td}  top5={top5}")

Steering vector norm: 4.4062
  alpha=  0.5  top digit=1  top5=['1', '9', '2', '8', '0']
  alpha=  1.0  top digit=1  top5=['1', '9', '2', '8', '0']
  alpha=  2.0  top digit=1  top5=['1', '9', '2', '8', '0']
  alpha=  5.0  top digit=1  top5=['1', '9', '8', '2', '0']
  alpha= 10.0  top digit=1  top5=['1', '9', '8', '2', '5']


In [30]:
# ── Diagnostic 1: Probe dissociation check for Qwen ──────────────────────
# Is the count encoded but not used (like Llama 1B)?
# Or is it simply not encoded until late layers?
# Key question: at the layers where the wrong answer locks in (L22-L25),
# is the probe R2 high enough to claim dissociation?

import json
import numpy as np

with open("qwen1.5B/probe_results_qwen2.5_1.5b_instruct.json") as f:
    probe_data = json.load(f)

maes_rep  = probe_data["repeated"]["maes"]
r2s_rep   = probe_data["repeated"]["r2s"]
maes_uniq = probe_data["unique"]["maes"]
r2s_uniq  = probe_data["unique"]["r2s"]

print("=" * 65)
print("PROBE DISSOCIATION DIAGNOSTIC — Qwen2.5-1.5B")
print("=" * 65)
print(f"\nKey question: is R2(repeated) high at the lock-in layers (L22-L25)?")
print(f"Lock-in layers for Qwen 1.5B: L22 (MLP writes '8'), L24 (MLP writes '8')")
print(f"\n{'Layer':>6}  {'R2(rep)':>9}  {'R2(uniq)':>9}  {'MAE(rep)':>9}  "
      f"{'MAE(uniq)':>10}  {'Dissociation?':>15}")
print("-" * 70)

LOCKIN_LAYERS = [20, 21, 22, 23, 24, 25, 26]
DISSOC_THRESHOLD = 0.95  # minimum R2 to claim count is encoded

for i in range(len(maes_rep)):
    label    = "embed" if i == 0 else f"L{i:02d}"
    r2r      = r2s_rep[i]
    r2u      = r2s_uniq[i]
    mar      = maes_rep[i]
    mau      = maes_uniq[i]
    is_lockin = i in LOCKIN_LAYERS
    # Dissociation = count encoded (R2 high) but model outputs wrong answer
    dissoc   = "YES" if r2r > DISSOC_THRESHOLD and is_lockin \
               else "weak" if r2r > 0.90 and is_lockin \
               else "NO" if is_lockin \
               else "-"
    marker   = " <-- LOCK-IN" if is_lockin else ""
    print(f"{label:>6}  {r2r:>9.4f}  {r2u:>9.4f}  {mar:>9.4f}  "
          f"{mau:>10.4f}  {dissoc:>15}{marker}")

# Summary
print(f"\nDissociation summary:")
lockin_r2s = [r2s_rep[i] for i in LOCKIN_LAYERS]
print(f"  Mean R2(repeated) at lock-in layers L20-L26: {np.mean(lockin_r2s):.4f}")
print(f"  Min  R2(repeated) at lock-in layers L20-L26: {np.min(lockin_r2s):.4f}")
print(f"  Llama 1B R2 at lock-in (L14): {0.9945:.4f}  (for reference)")
print(f"  Threshold for claiming dissociation: {DISSOC_THRESHOLD}")

if np.mean(lockin_r2s) > DISSOC_THRESHOLD:
    print(f"\n  VERDICT: Dissociation holds for Qwen — count encoded at lock-in layers")
elif np.mean(lockin_r2s) > 0.90:
    print(f"\n  VERDICT: Weak dissociation — count partially encoded at lock-in layers")
else:
    print(f"\n  VERDICT: No dissociation — count not encoded at lock-in layers")


# ── Diagnostic 2: Logit lens tokenizer check ─────────────────────────────
# Why does P3 output "10" correctly but "10" never appears in logit lens?
# Check: is "10" a single token in Qwen tokenizer?
# If "10" tokenizes to ["1", "0"] then get_top_digit skips it entirely.

print("\n" + "=" * 65)
print("LOGIT LENS TOKENIZER DIAGNOSTIC — Qwen2.5-1.5B")
print("=" * 65)
print("\nChecking digit tokenization (single token = usable in logit lens):")
print(f"{'Digit':>6}  {'Token IDs':>20}  {'Decoded':>15}  Single token?")
print("-" * 58)

for n in range(1, 21):
    ids     = tokenizer.encode(str(n), add_special_tokens=False)
    decoded = [tokenizer.decode([i]) for i in ids]
    single  = "YES" if len(ids) == 1 else f"NO — {len(ids)} tokens"
    print(f"{n:>6}  {str(ids):>20}  {str(decoded):>15}  {single}")

# This directly shows which digits the logit lens can track
# If "10" is multi-token, it explains why P3 logit lens never shows "10"
# despite the model outputting it correctly

print("\nImplication for logit lens:")
ids_10 = tokenizer.encode("10", add_special_tokens=False)
if len(ids_10) > 1:
    print(f"  '10' tokenizes to {ids_10} — multi-token, invisible to logit lens")
    print(f"  The logit lens top-digit tracking MISSES '10' entirely")
    print(f"  P3 outputs '10' correctly but logit lens can't show it")
    print(f"  This is a methodological limitation, not a model failure")
    print(f"  Logit lens results for Qwen should only be interpreted for")
    print(f"  single-token digits: {[n for n in range(1,21) if len(tokenizer.encode(str(n), add_special_tokens=False))==1]}")
else:
    print(f"  '10' is a single token — logit lens should track it")
    print(f"  P3 failure to show '10' is a genuine model behavior, not an artifact")

# ── Diagnostic 3: Direct logit check at final layer for P3 ───────────────
# What are the actual logits for "10" at the final layer of P3?
# Even if not top-1, where does "10" rank?

print("DIRECT LOGIT CHECK — P3 final layer, Qwen2.5-1.5B")
print("=" * 65)

remove_all_hooks(model_eager)

inputs = make_inputs_eager(PROMPTS["phase3_control"]["text"])

# Collect final hidden state via hook
final_hidden = {}
def hook_final(module, input, output):
    h = output[0] if isinstance(output, tuple) else output
    final_hidden["h"] = h[0, -1, :].detach().clone()

handle = model_eager.model.layers[-1].register_forward_hook(hook_final)
with torch.no_grad():
    model_eager(**inputs)
handle.remove()

# Project through norm + unembedding
normed = model_eager.model.norm(
    final_hidden["h"].unsqueeze(0).unsqueeze(0)
).squeeze()
logits = model_eager.lm_head.weight @ normed

# Check all digit tokens
print("\nLogit values for digit strings at final layer (P3):")
print(f"{'Digit':>6}  {'Token IDs':>15}  {'Logit':>10}  {'Rank':>8}  Single?")
print("-" * 55)

all_logits_sorted = logits.argsort(descending=True)

for n in range(1, 21):
    ids    = tokenizer.encode(str(n), add_special_tokens=False)
    single = len(ids) == 1
    if single:
        logit_val = logits[ids[0]].item()
        rank      = (all_logits_sorted == ids[0]).nonzero().item() + 1
        print(f"{n:>6}  {str(ids):>15}  {logit_val:>10.4f}  {rank:>8}  YES")
    else:
        print(f"{n:>6}  {str(ids):>15}  {'N/A':>10}  {'N/A':>8}  NO — multi-token")

# Top-20 tokens by logit at final layer
print(f"\nTop-20 tokens at final layer (P3):")
top20 = logits.topk(20)
for i, (val, idx) in enumerate(zip(top20.values, top20.indices)):
    tok = tokenizer.decode([idx])
    print(f"  {i+1:>3}. {repr(tok):<20}  logit={val.item():.4f}")

PROBE DISSOCIATION DIAGNOSTIC — Qwen2.5-1.5B

Key question: is R2(repeated) high at the lock-in layers (L22-L25)?
Lock-in layers for Qwen 1.5B: L22 (MLP writes '8'), L24 (MLP writes '8')

 Layer    R2(rep)   R2(uniq)   MAE(rep)   MAE(uniq)    Dissociation?
----------------------------------------------------------------------
 embed    -0.2100    -0.2100     3.0000      3.0000                -
   L01     0.9023     0.7457     0.8342      1.1863                -
   L02     0.7336     0.5150     1.2315      1.5103                -
   L03     0.7253     0.7556     1.3739      1.1317                -
   L04     0.7069     0.8285     1.3194      0.8274                -
   L05     0.7687     0.8569     1.1719      0.7768                -
   L06     0.7239     0.8769     1.3072      0.7244                -
   L07     0.6874     0.8632     1.3564      0.7938                -
   L08     0.7076     0.8470     1.3389      0.8448                -
   L09     0.8044     0.8715     1.1035      0.7284

In [29]:
def get_full_attentions(prompt_text):
    inputs  = make_inputs_eager(prompt_text)
    n_layers = model_eager.config.num_hidden_layers
    all_attentions = []

    def make_attn_hook():
        def hook(module, input, output):
            # output is tuple: (hidden_state, attn_weights, ...)
            # attn_weights shape: (1, n_heads, seq_len, seq_len)
            if isinstance(output, tuple) and len(output) > 1:
                attn = output[1]
                if attn is not None and isinstance(attn, torch.Tensor):
                    all_attentions.append(attn.detach().cpu().float())
        return hook

    handles = []
    for layer in model_eager.model.layers:
        handles.append(layer.self_attn.register_forward_hook(make_attn_hook()))

    with torch.no_grad():
        model_eager(**inputs)

    for h in handles:
        h.remove()

    if len(all_attentions) == 0:
        print("  Warning: no attention weights captured via hooks.")
        print("  Trying with output_attentions=True on model config...")
        model_eager.config.output_attentions = True
        with torch.no_grad():
            out = model_eager(**inputs)
        model_eager.config.output_attentions = False
        if out.attentions:
            all_attentions = [a.detach().cpu().float() for a in out.attentions]

    if len(all_attentions) == 0:
        raise RuntimeError(
            "Could not capture attention weights. "
            "Try reloading model_eager with attn_implementation='eager'."
        )

    # Stack: (n_layers, 1, n_heads, seq_len, seq_len) -> squeeze batch
    attentions = torch.stack(all_attentions).squeeze(1)
    print(f"  Captured {len(all_attentions)} layers, shape: {attentions.shape}")
    return attentions, inputs["input_ids"][0]


# Test
remove_all_hooks(model_eager)
print("Testing attention extraction...")
attn_test, ids_test = get_full_attentions(PROMPTS["phase1_baseline"]["text"])
print(f"Success. Shape: {attn_test.shape}")

Testing attention extraction...
  Captured 36 layers, shape: torch.Size([36, 16, 62, 62])
Success. Shape: torch.Size([36, 16, 62, 62])


In [31]:
# ── Attention analysis: P1 vs P2 — does any head attend to banana? ────────
remove_all_hooks(model_eager)

# FIXED: hook-based attention extraction — works with sdpa
def get_full_attentions(prompt_text):
    inputs        = make_inputs_eager(prompt_text)
    all_attentions = []

    def make_attn_hook():
        def hook(module, input, output):
            if isinstance(output, tuple) and len(output) > 1:
                attn = output[1]
                if attn is not None and isinstance(attn, torch.Tensor):
                    all_attentions.append(attn.detach().cpu().float())
        return hook

    handles = []
    for layer in model_eager.model.layers:
        handles.append(layer.self_attn.register_forward_hook(make_attn_hook()))

    with torch.no_grad():
        model_eager(**inputs)

    for h in handles:
        h.remove()

    attentions = torch.stack(all_attentions).squeeze(1)
    return attentions, inputs["input_ids"][0]


# ── Rest of analysis unchanged from here ─────────────────────────────────
def find_word_positions_3b(tokens, phase_key):
    content_words = {
        "phase1_baseline": {"apple"},
        "phase2_anomaly" : {"apple", "banana"},
    }
    valid     = content_words[phase_key]
    colon_idx = max(i for i, t in enumerate(tokens) if t.strip() == ":")
    positions = []
    for i in range(colon_idx + 1, len(tokens)):
        if tokens[i].strip() in valid:
            positions.append(i)
        elif "." in tokens[i] and positions:
            break
    return positions

print("Extracting attentions for P1 and P2...")
attn_p1, ids_p1 = get_full_attentions(PROMPTS["phase1_baseline"]["text"])
attn_p2, ids_p2 = get_full_attentions(PROMPTS["phase2_anomaly"]["text"])

tokens_p1 = [tokenizer.decode([t]) for t in ids_p1]
tokens_p2 = [tokenizer.decode([t]) for t in ids_p2]

word_pos_p1 = find_word_positions_3b(tokens_p1, "phase1_baseline")
word_pos_p2 = find_word_positions_3b(tokens_p2, "phase2_anomaly")

banana_pos_in_list = None
banana_seq_pos     = None
for list_idx, seq_pos in enumerate(word_pos_p2):
    if tokens_p2[seq_pos].strip() == "banana":
        banana_pos_in_list = list_idx
        banana_seq_pos     = seq_pos
        break

apple_pos_p2 = [p for p in word_pos_p2 if tokens_p2[p].strip() == "apple"]

print(f"P1 word positions : {word_pos_p1}")
print(f"P2 word positions : {word_pos_p2}")
print(f"Banana at list pos {banana_pos_in_list}, seq pos {banana_seq_pos}")
print(f"P2 apple positions: {apple_pos_p2}")

n_layers = attn_p1.shape[0]
n_heads  = attn_p1.shape[1]

print(f"\n{'='*72}")
print("ATTENTION TO BANANA vs APPLE — P2 (per layer, mean over heads)")
print(f"{'='*72}")
print(f"  {'Layer':>6}  {'Attn→banana':>12}  {'Attn→apple(mean)':>17}  "
      f"{'Ratio b/a':>10}  Ignored?")
print("  " + "-" * 60)

attention_results = []

for layer_idx in range(n_layers):
    last_tok_attn  = attn_p2[layer_idx, :, -1, :]
    mean_attn      = last_tok_attn.mean(dim=0)
    word_attn      = mean_attn[word_pos_p2]
    word_attn_norm = word_attn / (word_attn.sum() + 1e-9)

    b_idx          = word_pos_p2.index(banana_seq_pos)
    banana_attn    = word_attn_norm[b_idx].item()
    apple_indices  = [word_pos_p2.index(p) for p in apple_pos_p2]
    apple_attn_mean = word_attn_norm[apple_indices].mean().item()
    ratio           = banana_attn / (apple_attn_mean + 1e-9)
    ignored         = "YES" if ratio < 0.5 else "NO" if ratio > 1.5 else "partial"

    print(f"  L{layer_idx+1:02d}    {banana_attn:>12.4f}  {apple_attn_mean:>17.4f}  "
          f"{ratio:>10.3f}  {ignored}")
    attention_results.append({
        "layer": layer_idx + 1, "banana_attn": banana_attn,
        "apple_attn_mean": apple_attn_mean, "ratio": ratio, "ignored": ignored,
    })

# Per-head at most ignoring layers
print(f"\n{'='*72}")
print("PER-HEAD ANALYSIS — 3 layers where banana is most ignored")
print(f"{'='*72}")

sorted_layers  = sorted(attention_results, key=lambda x: x["ratio"])[:3]
ignored_layers = [r["layer"] for r in sorted_layers]
print(f"Most ignoring layers: {ignored_layers}")

for layer_idx in [l-1 for l in ignored_layers]:
    last_tok_attn = attn_p2[layer_idx, :, -1, :]
    print(f"\n  L{layer_idx+1:02d} — per-head banana vs apple:")
    print(f"  {'Head':>5}  {'→banana':>9}  {'→apple':>9}  {'Ratio':>7}  Role")
    print("  " + "-" * 45)
    for head_idx in range(n_heads):
        head_attn      = last_tok_attn[head_idx]
        word_attn      = head_attn[word_pos_p2]
        word_attn_norm = word_attn / (word_attn.sum() + 1e-9)
        b_idx          = word_pos_p2.index(banana_seq_pos)
        b_attn         = word_attn_norm[b_idx].item()
        a_idxs         = [word_pos_p2.index(p) for p in apple_pos_p2]
        a_mean         = word_attn_norm[a_idxs].mean().item()
        ratio          = b_attn / (a_mean + 1e-9)
        role           = "IGNORES" if ratio < 0.3 \
                         else "ATTENDS" if ratio > 2.0 else "-"
        if role != "-":
            print(f"  H{head_idx:02d}    {b_attn:>9.4f}  {a_mean:>9.4f}  "
                  f"{ratio:>7.3f}  {role}")

# Entropy comparison
print(f"\n{'='*72}")
print("ENTROPY COMPARISON — P1 vs P2 word-list attention")
print(f"{'='*72}")
print(f"  {'Layer':>6}  {'H(P1)':>10}  {'H(P2)':>10}  {'ΔH':>8}")
print("  " + "-" * 40)

for layer_idx in range(n_layers):
    a1   = attn_p1[layer_idx, :, -1, :].mean(dim=0)
    wa1  = a1[word_pos_p1]; wa1 = wa1 / (wa1.sum() + 1e-9)
    h1   = -(wa1 * (wa1 + 1e-9).log()).sum().item()

    a2   = attn_p2[layer_idx, :, -1, :].mean(dim=0)
    wa2  = a2[word_pos_p2]; wa2 = wa2 / (wa2.sum() + 1e-9)
    h2   = -(wa2 * (wa2 + 1e-9).log()).sum().item()

    delta  = h2 - h1
    marker = " <-- P2 concentrated" if delta < -0.1 \
             else " <-- P2 spread" if delta > 0.1 else ""
    print(f"  L{layer_idx+1:02d}    {h1:>10.4f}  {h2:>10.4f}  {delta:>8.4f}{marker}")

with open("banana_attention_qwen3b.json", "w") as f:
    json.dump({"model": MODEL_NAME, "attention_results": attention_results}, f, indent=2)
print("\nSaved: banana_attention_qwen3b.json")

Extracting attentions for P1 and P2...
P1 word positions : [37, 38, 39, 40, 41, 42, 43, 44, 45, 46]
P2 word positions : [37, 38, 39, 40, 41, 42, 43, 44, 45, 46]
Banana at list pos 4, seq pos 41
P2 apple positions: [37, 38, 39, 40, 42, 43, 44, 45, 46]

ATTENTION TO BANANA vs APPLE — P2 (per layer, mean over heads)
   Layer   Attn→banana   Attn→apple(mean)   Ratio b/a  Ignored?
  ------------------------------------------------------------
  L01          0.0950             0.1006       0.945  partial
  L02          0.0852             0.1016       0.838  partial
  L03          0.1071             0.0992       1.079  partial
  L04          0.0411             0.1065       0.386  YES
  L05          0.2242             0.0862       2.601  NO
  L06          0.0938             0.1007       0.931  partial
  L07          0.1904             0.0900       2.117  NO
  L08          0.2276             0.0858       2.652  NO
  L09          0.0961             0.1004       0.957  partial
  L10          0.09

In [32]:
# Save banana attention summary
banana_summary = {
    "model"   : MODEL_NAME,
    "finding" : "banana over-attended in 10/36 layers (ratio > 1.5), not ignored",
    "key_layers": {
        "over_attended" : "L05, L07, L08, L11, L12, L16, L17, L18, L20, L35",
        "ignored"       : "L04, L28 (mean over heads)",
        "L28_detail"    : "late-layer collapse — 10/16 heads ignore banana, 1 head (H11) attends strongly",
    },
    "entropy": {
        "L18" : "P2 more concentrated — attention collapsing as model commits to wrong answer",
        "L34" : "P2 more concentrated — same pattern near final layers",
        "L09" : "P2 more spread — banana causing broader attention in early layers",
    },
    "interpretation": (
        "Failure is not attention blindness. Banana is detected at attention level "
        "but detection signal is not routed to corrected count output. "
        "Routing failure consistent with Llama finding but at attention level "
        "rather than residual stream level."
    ),
}
with open("banana_attention_summary_qwen3b.json", "w") as f:
    json.dump(banana_summary, f, indent=2)
print("Saved. Move to Qwen 7B.")

Saved. Move to Qwen 7B.
